In [ ]:
# Import libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import datetime
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import cmocean

# To avoid warning messages
import warnings
warnings.filterwarnings('ignore')

# NPP CALCULATION

**Standard VGPM**

Variables used: chl-a, sst, and par

In [ ]:
# Load the file

data = r'C:\Users\ratna\OneDrive\Documents\NPP\Temporary output\combined file with all vars1.nc'
data_all_vars = xr.open_dataset(data)
data_all_vars

In [ ]:
epu_shp = r'\Users\ratna\OneDrive\Documents\NPP\EPUs\EPUs.shp'
gdf = gpd.read_file(epu_shp)

gdf = gdf.to_crs(epsg=4326)

**Calculate chl_tot**

Please kindly add short definion of this element

chl <  1.0
chl_tot = 38.0 * chl^0.425

chl > 1.0
chl_tot = 40.2 * chl^0.507

In [ ]:
chl = data_all_vars['CHL']

chl_tot = xr.where(chl < 1.0, 38.0 * chl**0.425, 40.2 * chl**0.507)

# list or print (chl_tot)

**Calculate Euphotic Depth (z_eu)**

Model used: Morel's case I model from satellite surface chlorophyll-a data

In [ ]:
# when z_eu > 102 m
z_eu1 = 200.0 * chl_tot**-0.293

#list(z_eu1)

In [ ]:
# when z_eu < 102 m

z_eu2 = xr.where(z_eu1 < 102.0, 568.2*chl_tot**-0.746, z_eu1)

#list(z_eu2)

In [ ]:
# when z_eu < 102 m

z_eu3 = 568.2*chl_tot**-0.746

#z_eu3

In [ ]:
# when z_eu > 102 m

z_eu4 = xr.where(z_eu1 > 102.0, 568.2*chl_tot**-0.746, z_eu3)

#z_eu4

**Calculate the Pb_opt from satellite sea surface temperature (sst)**

In [ ]:
sst = data_all_vars['SST']

pb_opt = xr.where(sst < -10.0, 0.00,
          xr.where(sst < -1.0, 1.13,
          xr.where(sst > 28.5, 4.00,
          (1.2956 + 2.749e-1*sst + 6.17e-2* sst**2 - 2.05e-2* sst**3 +
           2.462e-3* sst**4 - 1.348e-4* sst**5 + 3.4132e-6* sst**6 -
           3.27e-8* sst**7))))

#list(pb_opt)

In [ ]:
#pb_opt.to_netcdf(r'C:\Users\ratna\OneDrive\Documents\NPP\Temporary output\pb_opt1.nc')

**Determine the day length**

Input: latitude and julian day

In [ ]:
time = data_all_vars['time']
lat = data_all_vars['latitude']
lon = data_all_vars['longitude']
# pd.to_datetime(time) to show date

In [ ]:
new_time = pd.to_datetime(time) + pd.DateOffset(days=14)
day = new_time.dayofyear

In [ ]:
day_np = day.to_numpy()

day_np

In [ ]:
lon_grid, lat_grid = np.meshgrid(lon, lat)
day_expanded = np.expand_dims(day_np, axis=(1, 2))
day_broadcasted = np.broadcast_to(day_expanded, (len(time), len(lat), len(lon)))

print(day_broadcasted.shape)

In [ ]:
day_expanded1 = day_np[:, np.newaxis, np.newaxis]
day_broadcasted1 = np.broadcast_to(day_expanded, (len(time), len(lat), len(lon)))

print(day_broadcasted1.shape)

In [ ]:
lat_expanded = lat.values[np.newaxis, :, np.newaxis]
lat_expanded.shape

In [ ]:
import math
def Daylight(lat_expanded, day_broadcasted):
    P = np.arcsin(0.39795 * np.cos(0.2163108 + 2 * np.arctan(0.9671396 * np.tan(0.00860 * (day_broadcasted - 186)))))
    pi = np.pi
    daylightamount = 24 - (24 / pi) * np.arccos(
    (np.sin((0.8333 * pi / 180) + np.sin(lat_expanded * pi / 180) * np.sin(P)) / (np.cos(lat_expanded * pi / 180) * np.cos(P))))

    return daylightamount

In [ ]:
daylength = Daylight(lat_expanded, day_broadcasted)

daylength

**Calculate the irradiance function**

0.66125 * (PAR / PAR + 4.1)

In [ ]:
par = data_all_vars['PAR']

irr_func = 0.66125 * par / (par + 4.1)

**Calculate NPP**

In [ ]:
test = chl * z_eu4 * pb_opt * daylength * irr_func

In [ ]:
npp_vgpm = chl * z_eu2 * pb_opt * daylength * irr_func  # unit in mgC/m2/day

**Modified (Eppley) VGPM**

Variables used: chl-a, sst, and par

This model used basic model structure and parameterization of the standard VGPM but replaced the polynomial description of Pb_opt with the exponential relationship. 

In [ ]:
pb_opt_mod = 1.54*10**(0.0275*sst-0.07)

In [ ]:
#pb_opt_mod.to_netcdf(r'C:\Users\ratna\OneDrive\Documents\NPP\Temporary output\pb_opt2.nc')

In [ ]:
npp_eppley = chl * z_eu2 * pb_opt_mod * daylength * irr_func  # unit in mgC/m2/day

In [ ]:
vgpm_mon = npp_vgpm.groupby('time.month').mean(skipna=True)
vgpm_mon[0,:,:].plot(cmap= cmocean.cm.algae, vmax=550)

In [ ]:
vgpm_mon = npp_vgpm.groupby('time.month').mean(skipna=True)/1000

print(vgpm_mon.min().values)
print(vgpm_mon.max().values)

In [ ]:
# Initialize name of the months
month_name = [
    "January", "February", "March", "April", "May", "June", 
    "July", "August", "September", "October", "November", "December"
]

In [ ]:
dataplot = vgpm_mon[0, :, :]
plt.figure(figsize=(8, 6))
plt.pcolormesh(vgpm_mon.longitude, vgpm_mon.latitude, dataplot, cmap='nipy_spectral_r', vmin=0, vmax=550)
plt.colorbar()
plt.title('Month 1: January')
plt.show()

In [ ]:
# Define number of rows and columns for the multi-panel plots
nrows = 3
ncols = 4

# Define the figure and each axis for the 3 rows and 4 columns
fig, axs = plt.subplots(nrows = nrows, ncols = ncols,
                        subplot_kw = {'projection': ccrs.PlateCarree()},
                        figsize = (20,15), facecolor='w')

plt.subplots_adjust(bottom=0.15, top=0.96, left=0.04, right=0.99,
                    wspace=0.15, hspace=0.02)

#gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.3, wspace=0.2)

axs = axs.flatten()

# Dummy plot element for the shapefile (used in the legend)
shp_legend = mlines.Line2D([], [], color='red', label='EPU boundaries', linewidth=1)

for i in range(1, 13):
    vgpm_plot = vgpm_mon[i-1, :, :] # Remember that in Python, the data index starts at 0, but the subplot index start at 1.
    ax = axs[i-1]   # Access the correct subplot
    
    p = ax.pcolormesh(vgpm_mon.longitude, vgpm_mon.latitude, vgpm_plot, vmin=0, vmax=1.2, cmap = cmocean.cm.algae, zorder=1, alpha=0.8)
    
    ax.set_xlim(-78, -40)
    ax.set_ylim(35, 65)
    ax.set_title(month_name[i-1], fontsize = 13, fontweight = 'bold', color = 'k')
    ax.tick_params(axis='both', labelsize=11)
    ax.set_xticks(ax.get_xticks())
    ax.set_yticks(ax.get_yticks())
    if i % ncols == 1: # Add ylabel for the very left subplots
        ax.set_ylabel('Latitude', fontsize = 11, fontweight = 'bold')
    if i > ncols*(nrows-1): # Add xlabel for the bottom row subplots
        ax.set_xlabel('Longitude', fontsize = 11, fontweight = 'bold')

    # Add the shapefile geometries
    ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(),
                      facecolor='none', edgecolor='red', linewidth=0.3, zorder=2)
    
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=2)
    ax.add_feature(cfeature.BORDERS, facecolor='k', linestyle='-.',  linewidth=1, zorder=2)
    ax.coastlines(resolution='50m', linewidth=1, zorder=3)   # Add coastlines 

# Add a colorbar at the bottom:
cbar_ax = fig.add_axes([1.025, 0.25, 0.018, 0.5])   # 0.25 left, 0.1 bottom, width, height
cbar = plt.colorbar(p, cax=cbar_ax, orientation='vertical', extend = 'max',)  # orientation = vertical or horizontal
cbar.ax.tick_params(labelsize=11)
#cbar.set_label(label='Net Primary Productivity ($\mathrm{mgC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=14)

# Add the legend with the shapefile entry
fig.legend(handles=[shp_legend], loc='lower left', ncol=2, fontsize=12, bbox_to_anchor=(1, 0.1))

# Add title
fig.suptitle('Net Primary Productivity ($\mathrm{mgC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=20, y=1.0)

# Now we can save a high resolution (300dpi) version of the figure:
#plt.savefig('VGPM_monthly climatology_in g_300.png', format = 'png', dpi = 300, bbox_inches='tight')

plt.show()

In [ ]:
eppley_mon = npp_eppley.groupby('time.month').mean(skipna=True)/1000

print(eppley_mon.min().values)
print(eppley_mon.max().values)

In [ ]:
# Define number of rows and columns for the multi-panel plots
nrows = 3
ncols = 4

# Define the figure and each axis for the 3 rows and 4 columns
fig, axs = plt.subplots(nrows = nrows, ncols = ncols,
                        subplot_kw = {'projection': ccrs.PlateCarree()},
                        figsize = (20,15), facecolor='w')

plt.subplots_adjust(bottom=0.15, top=0.96, left=0.04, right=0.99,
                    wspace=0.15, hspace=0.02)

#gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.3, wspace=0.2)

axs = axs.flatten()

# Dummy plot element for the shapefile (used in the legend)
shp_legend = mlines.Line2D([], [], color='red', label='EPU boundaries', linewidth=1)

for i in range(1, 13):
    eppley_plot = eppley_mon[i-1, :, :] # Remember that in Python, the data index starts at 0, but the subplot index start at 1.
    ax = axs[i-1]   # Access the correct subplot
    
    p = ax.pcolormesh(eppley_mon.longitude, eppley_mon.latitude, eppley_plot, vmax = 1.2, vmin = 0, cmap = cmocean.cm.algae, zorder=1, alpha=0.8)
    
    ax.set_xlim(-78, -40)
    ax.set_ylim(35, 65)
    ax.set_title(month_name[i-1], fontsize = 13, fontweight = 'bold', color = 'k')
    ax.tick_params(axis='both', labelsize=11)
    ax.set_xticks(ax.get_xticks())
    ax.set_yticks(ax.get_yticks())
    if i % ncols == 1: # Add ylabel for the very left subplots
        ax.set_ylabel('Latitude', fontsize = 11, fontweight = 'bold')
    if i > ncols*(nrows-1): # Add xlabel for the bottom row subplots
        ax.set_xlabel('Longitude', fontsize = 11, fontweight = 'bold')

    # Add the shapefile geometries
    ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(),
                      facecolor='none', edgecolor='red', linewidth=0.3, zorder=2)
    
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=2)
    ax.add_feature(cfeature.BORDERS, facecolor='k', linestyle='-.',  linewidth=1, zorder=2)
    ax.coastlines(resolution='50m', linewidth=1, zorder=3)   # Add coastlines 

# Add a colorbar at the bottom:
cbar_ax = fig.add_axes([1.025, 0.25, 0.018, 0.5])   # 0.25 left, 0.1 bottom, width, height
cbar = plt.colorbar(p, cax=cbar_ax, orientation='vertical', extend = 'max',)  # orientation = vertical or horizontal
cbar.ax.tick_params(labelsize=11)
#cbar.set_label(label='Net Primary Productivity ($\mathrm{mgC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=14)

# Add the legend with the shapefile entry
fig.legend(handles=[shp_legend], loc='lower left', ncol=2, fontsize=12, bbox_to_anchor=(1, 0.1))

# Add title
fig.suptitle('Net Primary Productivity ($\mathrm{gC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=20, y=1.0)

# Now we can save a high resolution (300dpi) version of the figure:
#plt.savefig('Eppley_monthly climatology_in g_300.png', format = 'png', dpi = 300, bbox_inches='tight')

plt.show()

In [ ]:
# Choose desired variables
vgpm = npp_vgpm
evgpm = npp_eppley

# Combine more variables
npp_combined = xr.Dataset({
    'VGPM': vgpm,
    'EVGPM': evgpm,
})

npp_combined
# Save to new netcdf file
#npp_combined.to_netcdf(r'C:\Users\nsuhita\OneDrive - Marine Institute of Memorial University\Documents\NPP\Temporary output\npp_chl1.nc')

In [ ]:
data_pp = 'Raw Data1\cmems_oc_nwa_bgc-pp_multi4km.nc'
pp = xr.open_dataset(data_pp)

In [ ]:
ds_pp = pp['PP']
ds_pp

In [ ]:
pp_mon = ds_pp.groupby('time.month').mean(skipna=True)

pp_mon

In [ ]:
# Define number of rows and columns for the multi-panel plots
nrows = 3
ncols = 4

# Define the figure and each axis for the 3 rows and 4 columns
fig, axs = plt.subplots(nrows = nrows, ncols = ncols,
                        subplot_kw = {'projection': ccrs.PlateCarree()},
                        figsize = (20,15), facecolor='w')

plt.subplots_adjust(bottom=0.15, top=0.96, left=0.04, right=0.99,
                    wspace=0.15, hspace=0.02)

#gs = gridspec.GridSpec(nrows, ncols, figure=fig, hspace=0.3, wspace=0.2)

axs = axs.flatten()

# Dummy plot element for the shapefile (used in the legend)
shp_legend = mlines.Line2D([], [], color='red', label='EPU boundaries', linewidth=1)

for i in range(1, 13):
    pp_plot = pp_mon[i-1, :, :] # Remember that in Python, the data index starts at 0, but the subplot index start at 1.
    ax = axs[i-1]   # Access the correct subplot
    
    p = ax.pcolormesh(pp_mon.longitude, pp_mon.latitude, pp_plot, vmin=0, vmax=1200, cmap = cmocean.cm.algae, zorder=1, alpha=0.8)
    
    ax.set_xlim(-78, -40)
    ax.set_ylim(35, 65)
    ax.set_title(month_name[i-1], fontsize = 13, fontweight = 'bold', color = 'k')
    ax.tick_params(axis='both', labelsize=11)
    ax.set_xticks(ax.get_xticks())
    ax.set_yticks(ax.get_yticks())
    if i % ncols == 1: # Add ylabel for the very left subplots
        ax.set_ylabel('Latitude', fontsize = 11, fontweight = 'bold')
    if i > ncols*(nrows-1): # Add xlabel for the bottom row subplots
        ax.set_xlabel('Longitude', fontsize = 11, fontweight = 'bold')

    # Add the shapefile geometries
    ax.add_geometries(gdf.geometry, crs=ccrs.PlateCarree(),
                      facecolor='none', edgecolor='red', linewidth=0.3, zorder=2)
    
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=2)
    ax.coastlines(resolution='50m', linewidth=1, zorder=3)   # Add coastlines 

# Add a colorbar at the bottom:
cbar_ax = fig.add_axes([1.025, 0.25, 0.018, 0.5])   # 0.25 left, 0.1 bottom, width, height
cbar = plt.colorbar(p, cax=cbar_ax, orientation='vertical', extend = 'max',)  # orientation = vertical or horizontal
cbar.ax.tick_params(labelsize=11)
#cbar.set_label(label='Net Primary Productivity ($\mathrm{mgC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=14)

# Add the legend with the shapefile entry
fig.legend(handles=[shp_legend], loc='lower left', ncol=2, fontsize=12, bbox_to_anchor=(1, 0.1))

# Add title
fig.suptitle('Net Primary Productivity ($\mathrm{mgC}$ $\mathrm{m^{-2}}$ $\mathrm{day^{-1}}$)', color = 'k', size=20, y=1.0)

# Now we can save a high resolution (300dpi) version of the figure:
#plt.savefig('pp_monthly climatology_in g_300.png', format = 'png', dpi = 300, bbox_inches='tight')

plt.show()

In [ ]:
# annual mean of chl

chl = data_all_vars['CHL']
annual_chl = chl.resample(time='YS').mean(dim=['time', 'latitude', 'longitude'], skipna=True)

In [ ]:
annual_chl

In [ ]:
clim_chl = chl.mean(dim=['time','latitude','longitude'], skipna=True)

In [ ]:
clim_chl

In [ ]:
anom_chl = annual_chl-clim_chl

In [ ]:
anom_chl

In [ ]:
annual_pb_opt = pb_opt.resample(time='YS').mean(dim=['time', 'latitude', 'longitude'], skipna=True)
annual_pb_opt

In [ ]:
clim_pb_opt = pb_opt.mean(dim=['time','latitude','longitude'], skipna=True)
clim_pb_opt

In [ ]:
anom_pb_opt = annual_pb_opt - clim_pb_opt
anom_pb_opt

In [ ]:
annual_pb_mod = pb_opt_mod.resample(time='YS').mean(dim=['time', 'latitude', 'longitude'], skipna=True)
annual_pb_mod

In [ ]:
clim_pb_mod = pb_opt_mod.mean(dim=['time','latitude','longitude'], skipna=True)
clim_pb_mod

In [ ]:
anom_pb_mod = annual_pb_mod - clim_pb_mod
anom_pb_mod

## Calculation of each term

term 1 = clim_chl * annual_pb_opt
term 2 = clim_chl * anom_pb_opt
term 3 = anom_chl * clim_pb_opt
term 4 = anom_chl * anom_pb_opt

In [ ]:
# Relative contribution
term1 = clim_chl * clim_pb_opt

term2 = clim_chl * anom_pb_opt

term3 = anom_chl * clim_pb_opt

term4 = anom_chl * anom_pb_opt

In [ ]:
term1

In [ ]:
term4

In [ ]:
term2

In [ ]:
term3

In [ ]:
npp_rec = ((term2*0)+term1) + term2 + term3 + term4

In [ ]:
((term2*0)+term1)

In [ ]:
npp_rec.plot()

In [ ]:
years = np.unique(data_all_vars['time'].dt.year)

years

In [ ]:
fig, axes = plt.subplots()
ax=plt.plot(years, npp_rec, label='NPP reconstructed')
plt.plot(years, term1+(term2*0), label='mean NPP')
plt.plot(years, term1+term2, color='g', label='term2')
plt.plot(years, term1+term3, color='r', label='term3')
plt.plot(years, term1+term4, color='k', label='term4')
plt.legend()
#plt.savefig('plot_all_terms.png', format = 'png', dpi = 300)
plt.show()

In [ ]:
# Total variability

total_var = np.sum(npp_rec**2)

In [ ]:
# individual distribution
term1_new = term1+(term2*0)

contrib_term1 = np.sum(term1_new**2)/total_var * 100
contrib_term2 = np.sum(term2**2)/total_var * 100
contrib_term3 = np.sum(term3**2)/total_var * 100
contrib_term4 = np.sum(term4**2)/total_var * 100

In [ ]:
contrib_term1

In [ ]:
contrib_term2

In [ ]:
contrib_term3

In [ ]:
contrib_term4

In [ ]:
contrib_term1 + contrib_term2 + contrib_term3 + contrib_term4